# CalmFruits — неделя 3: единственная финальная оценка

Эта тетрадь запускается только после фиксации `reports/selected_configuration.json` и review экспериментального протокола. Первый успешный запуск читает официальный test ровно один раз; последующие запуски читают сохранённый итоговый результат.

In [1]:
from __future__ import annotations

import importlib.metadata
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from joblib import load
from scipy import sparse

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
assert (ROOT / 'src' / 'calmfruits').exists()
sys.path.insert(0, str(ROOT / 'src'))

from calmfruits.data import load_local_env, read_parquet_from_s3
from calmfruits.final import assert_final_input_contract, run_final_once
from calmfruits.provenance import write_manifest
from calmfruits.search import LexicalSearch
from calmfruits.tracking import log_evaluation_run

CATALOG_DIR = ROOT / 'artifacts' / 'catalog'
INDEX_DIR = ROOT / 'artifacts' / 'indexes'
FINAL_DIR = ROOT / 'artifacts' / 'final'
REPORT_DIR = ROOT / 'reports'
assert load_local_env(ROOT / '.env')
environment = pd.DataFrame({'value': {'python':sys.version.split()[0], 'pandas':importlib.metadata.version('pandas'), 'mlflow':importlib.metadata.version('mlflow'), 'device':'CPU', 'purpose':'one-time final test evaluation'}})
display(environment)


,value
python,3.12.6
pandas,2.2.3
mlflow,3.10.0
device,CPU
purpose,one-time final test evaluation


## 1. Проверка замороженной конфигурации

До чтения test подтверждаем, что validation-победитель, каталог и поисковые артефакты существуют и согласованы.

In [2]:
selected = json.loads((REPORT_DIR / 'selected_configuration.json').read_text())
assert selected['winner']['method'] == 'lexical', 'The frozen winner must be loaded by its declared method'
catalog = pd.read_parquet(CATALOG_DIR / 'evaluation_catalog.parquet')
lexical = LexicalSearch(catalog, load(INDEX_DIR / 'tfidf_vectorizer.joblib'), sparse.load_npz(INDEX_DIR / 'tfidf_matrix.npz'))
assert catalog.imt_id.is_unique and len(catalog) == selected['winner'].get('catalog_size', len(catalog))
frozen_selection = {'method': selected['winner']['method'], 'validation_run_id': selected['run_ids'][selected['winner']['method']], 'fingerprints': selected['fingerprints']}
write_manifest(FINAL_DIR / 'frozen_selection.json', frozen_selection)
display(pd.DataFrame({'value': frozen_selection}))


,value
method,lexical
validation_run_id,12fc2b3407e9429fb95babd7b7ebde01
fingerprints,{'baseline_manifest': '39515dd885e2c296fea59d3...


**Вывод по `frozen_selection`.** До чтения test зафиксирован validation-победитель TF-IDF, его validation MLflow run ID и fingerprints каталога, split и golden-set. Замороженная конфигурация совпадает с `selected_configuration.json`.

## 2. Одноразовый test-прогон

При первом запуске загружается test, проверяется отсутствие пересечения с train и вычисляется только замороженный TF-IDF. Каждая обработанная выдача добавляется в checkpoint; completed state предотвращает повторный прогон.

In [3]:
state_path = FINAL_DIR / 'final_test_state.json'
if state_path.exists() and json.loads(state_path.read_text())['status'] == 'completed':
    final_per_query = pd.read_csv(FINAL_DIR / 'per_query_metrics.csv')
    final_top10 = pd.read_csv(FINAL_DIR / 'top10_results.csv')
    final_summary = pd.read_csv(FINAL_DIR / 'summary.csv')
    final_test_mode = 'loaded_completed_checkpoint'
else:
    train = read_parquet_from_s3('queries_synthetic_train.parquet')
    test = read_parquet_from_s3('queries_synthetic_test.parquet')
    assert_final_input_contract(test, train)
    final_per_query, final_top10, final_summary = run_final_once(lexical.search_lexical, test, catalog.imt_id, frozen_selection, FINAL_DIR)
    final_test_mode = 'executed_once'
final_summary.to_csv(REPORT_DIR / 'final_test_metrics.csv', index=False)
display(pd.DataFrame({'mode':[final_test_mode], 'test_queries':[final_per_query.query_id.nunique()], 'catalog_size':[len(catalog)]}))
display(final_summary)


,mode,test_queries,catalog_size
0,loaded_completed_checkpoint,300,3280


,method,mrr,precision_at_1,recall_at_1,hit_rate_at_1,ndcg_at_1,precision_at_3,recall_at_3,hit_rate_at_3,ndcg_at_3,precision_at_5,recall_at_5,hit_rate_at_5,ndcg_at_5,precision_at_10,recall_at_10,hit_rate_at_10,ndcg_at_10
0,lexical,0.341848,0.203333,0.111212,0.203333,0.227619,0.184444,0.257135,0.39,0.337828,0.162,0.36004,0.49,0.382051,0.123,0.530079,0.636667,0.413974


**Вывод по `final_summary`.** Единственный test-прогон обработал 300 запросов в режиме `executed_once`; при последующем чистом запуске notebook загрузил completed checkpoint. TF-IDF получил NDCG@10 0,414, MRR 0,342, Recall@10 0,530 и HitRate@10 0,637 на каталоге из 3 280 карточек.

## 3. Финальные ошибки и рекомендации

Показываем три test-запроса с наименьшим NDCG@10 и сопоставляем выдачу с размеченными товарами.

In [4]:
worst_test_queries = final_per_query.nsmallest(3, 'ndcg_at_10')[['query_id','query_text','ndcg_at_10','mrr','reachable_relevant_share']]
worst_test_top10 = final_top10.merge(worst_test_queries[['query_id']], on='query_id', how='inner')
display(worst_test_queries)
display(worst_test_top10[['query_id','rank','imt_id','relevance','imt_name','subj_name','score']])


,query_id,query_text,ndcg_at_10,mrr,reachable_relevant_share
16,SYN_000080,свободные рубашки,0.0,0.090909,1.0
30,SYN_000113,футболки lovetex store,0.0,0.013699,1.0
52,SYN_000217,легкие костюмы,0.0,0.010638,1.0


,query_id,rank,imt_id,relevance,imt_name,subj_name,score
0,SYN_000080,1,10131886,0,Рубашка,Рубашки,0.217220
1,SYN_000080,2,10039626,0,Рубашка,Рубашки,0.183217
2,SYN_000080,3,10039630,0,Рубашка,Рубашки,0.182771
3,SYN_000080,4,10174222,0,Рубашка,Рубашки,0.143063
4,SYN_000080,5,10168709,0,Брюки,Брюки,0.134548
5,SYN_000080,6,10010258,0,Рубашка,Рубашки,0.130972
6,SYN_000080,7,10076550,0,Рубашка,Рубашки,0.121143
7,SYN_000080,8,10154156,0,Джинсы,Джинсы,0.118152
8,SYN_000080,9,10055377,0,Шорты Alpha Унисекс,Шорты,0.117153
9,SYN_000080,10,10018739,0,Женское платье в цветочный принт,Платья,0.116788


**Вывод по `worst_test_queries` и `worst_test_top10`.** Для «свободные рубашки», «футболки lovetex store» и «легкие костюмы» NDCG@10 равен 0 при `reachable_relevant_share=1`: релевантные товары доступны, но точные токены и брендовые/атрибутивные формулировки не дали им войти в top-10. Это подтверждает потребность в будущем улучшении перефразов и брендовых атрибутов, а не в расширении текущего каталога.

## 4. MLflow и проверка сохранения

Логируем финальную test-оценку замороженного победителя, сохраняя результаты и manifest как артефакты.

In [5]:
final_run_id_path = FINAL_DIR / 'mlflow_run_id.txt'
if final_run_id_path.exists():
    final_run_id = final_run_id_path.read_text().strip()
else:
    final_run_id = log_evaluation_run('lexical_final_test', {'method':'lexical','eval_split':'test','catalog_size':len(catalog),'relevance_threshold':2,'validation_run_id':frozen_selection['validation_run_id'],'selection_fingerprints':json.dumps(frozen_selection['fingerprints'], sort_keys=True)}, final_summary, [FINAL_DIR / 'per_query_metrics.csv', FINAL_DIR / 'summary.csv', FINAL_DIR / 'frozen_selection.json'])
    final_run_id_path.write_text(final_run_id + '\n')
display(pd.DataFrame({'final_mlflow_run_id':[final_run_id], 'final_state':[json.loads(state_path.read_text())['status']]}))


🏃 View run lexical_final_test at: https://mlflow-ds-20260620-97ae288888.infra.data-science.education-services.ru/#/experiments/1/runs/eea52cfcb9ef4bdd826a3a09287bfc2c
🧪 View experiment at: https://mlflow-ds-20260620-97ae288888.infra.data-science.education-services.ru/#/experiments/1


,final_mlflow_run_id,final_state
0,eea52cfcb9ef4bdd826a3a09287bfc2c,completed


**Вывод по final MLflow run.** Финальный run `eea52cfcb9ef4bdd826a3a09287bfc2c` имеет статус FINISHED и связан с validation run выбранного TF-IDF. `check_mlflow.py` подтверждает experiment и семь успешных runs с обязательными метриками и параметрами.